# Decile portfolios from CNN predictions

In this section, I construct 10 decile portfolios from the model's predicted probability of an up move (`pred_prob_up`).

The logic is:

1. Use `end_date` as the portfolio formation date.
2. Compute the realized 5-day forward return from `close_now` and `close_future`.
3. On each rebalance date, sort all stocks into 10 deciles based on `pred_prob_up`.
4. Compute the equal-weight return of each decile.
5. Annualize the mean return and Sharpe ratio.

This follows the general setup in Jiang (2023) and my supervisor's paper, where stocks are sorted into decile portfolios using model predictions, the portfolios are equal-weighted, and held for five trading days.

In [2]:
import pandas as pd
import numpy as np
import os

## 1. Load the CSV and check the required columns

The file contains one row per stock-image observation.  
The most important columns here are:

- `end_date`: the date on which the image ends and the portfolio is formed
- `pred_prob_up`: the model's predicted probability that the future return is positive
- `close_now`: the close price at portfolio formation
- `close_future`: the close price 5 trading days later

I first load the data and make sure the required columns are present.

In [ ]:
HEAD_FOLDER = "color_candlestick_rsi_14_quarter_panel" # indsæt nuværende Folder her


In [5]:
csv_path = os.path.join(HEAD_FOLDER, "predictions.csv") # for det gamle brug "test_predictions". For nye filer bare brug "predictions"

df = pd.read_csv(csv_path)

required_cols = [
    "ticker", "end_date", "close_now", "close_future", "pred_prob_up"
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Parse date
df["end_date"] = pd.to_datetime(df["end_date"])

# Keep only the columns we need here
df = df[["ticker", "end_date", "close_now", "close_future", "pred_prob_up"]].copy()

print(df.head())
print("\nNumber of rows:", len(df))
print("Number of unique end_date values:", df["end_date"].nunique())
print("Number of unique tickers:", df["ticker"].nunique())

   ticker   end_date  close_now  close_future  pred_prob_up
0   10078 2001-01-30    32.0625       27.8125      0.513003
1   10078 2001-02-06    27.8125       25.1250      0.542351
2   10078 2001-02-13    25.1250       19.6250      0.547051
3   10078 2001-02-21    19.6250       19.8750      0.569962
4   10078 2001-02-28    19.8750       22.0625      0.547666

Number of rows: 71336
Number of unique end_date values: 727
Number of unique tickers: 702


## 2. Compute the realized 5-day forward return

For each stock observation, I compute the realized holding-period return as

$$
R_{i,t \to t+5} = \frac{\text{close\_future}_{i,t}}{\text{close\_now}_{i,t}} - 1
$$

This is the return that each decile portfolio earns over the next 5 trading days.

In [6]:
# Basic cleaning
df = df.dropna(subset=["ticker", "end_date", "close_now", "close_future", "pred_prob_up"]).copy()
df = df[(df["close_now"] > 0) & (df["close_future"] > 0)].copy()

# Remove duplicate stock-date observations if any
df = df.sort_values(["end_date", "ticker"]).drop_duplicates(subset=["ticker", "end_date"])

# Realized 5-day forward return
df["forward_return_5d"] = df["close_future"] / df["close_now"] - 1

print(df[["ticker", "end_date", "close_now", "close_future", "pred_prob_up", "forward_return_5d"]].head())

     ticker   end_date  close_now  close_future  pred_prob_up  \
0     10078 2001-01-30    32.0625       27.8125      0.513003   
146   10104 2001-01-30    30.3125       27.6250      0.510453   
292   10107 2001-01-30    63.3750       62.5625      0.440380   
437   10108 2001-01-30    49.9900       49.0600      0.490912   
756   10145 2001-01-30    47.3700       49.2500      0.520752   

     forward_return_5d  
0            -0.132554  
146          -0.088660  
292          -0.012821  
437          -0.018604  
756           0.039688  


## 3. Select portfolio formation dates

The non-overlapping image sequences were generated separately for each stock. As a result,
image endpoints are not perfectly synchronized across stocks. The data therefore contain a
dominant set of formation dates with broad cross-sectional coverage, as well as off-cycle
dates with relatively few available stocks.

To construct sufficiently broad decile portfolios, I retain formation dates with at least
300 available stocks. In the current sample, the retained dates contain at least 333 stocks
and follow the dominant non-overlapping five-trading-day sequence.

The retained stocks on each formation date are subsequently used to construct the decile
portfolios.

In [7]:
# Number of available stocks on each formation date
counts_per_date = (
    df.groupby("end_date")["ticker"]
      .nunique()
      .sort_index()
)

# Require sufficiently broad cross-sectional coverage.
# With 10 deciles, 300 stocks corresponds to at least about 30 stocks per decile.
MIN_STOCKS = 300

rebalance_dates = counts_per_date[
    counts_per_date >= MIN_STOCKS
].index.sort_values()

# Keep only observations belonging to the retained formation dates
weekly_df = df[
    df["end_date"].isin(rebalance_dates)
].copy()

# Summary of the retained portfolio sample
stocks_per_date = (
    weekly_df.groupby("end_date")["ticker"]
             .nunique()
)

print("Total formation dates in full sample:", df["end_date"].nunique())
print("Rebalance dates retained:", len(rebalance_dates))
print("Rows retained:", len(weekly_df))

print("\nStocks per retained formation date:")
print(stocks_per_date.describe())

Total formation dates in full sample: 727
Rebalance dates retained: 146
Rows retained: 54625

Stocks per retained formation date:
count    146.000000
mean     374.143836
std       33.228776
min      333.000000
25%      346.250000
50%      364.500000
75%      390.000000
max      461.000000
Name: ticker, dtype: float64


Since you want to check the retained formation dates after the cleaning/filtering, insert the check directly after the Section 3 code cell that creates rebalance_dates and weekly_df, and before Section 4 where you assign deciles.

One important detail: you want to check 5 trading periods apart, not simply (date2 - date1).days == 5, because weekends and holidays make calendar-day differences vary.

In [8]:
# Check that consecutive retained formation dates are 5 trading-date positions apart

all_dates = pd.DatetimeIndex(
    sorted(pd.to_datetime(df["end_date"].unique()))
)

date_position = pd.Series(
    np.arange(len(all_dates)),
    index=all_dates
)

rebalance_positions = date_position.loc[rebalance_dates]
gaps = rebalance_positions.diff().dropna()

print("Gap distribution between consecutive rebalance dates:")
print(gaps.value_counts().sort_index())

if (gaps == 5).all():
    print("\nAll consecutive rebalance dates are 5 trading-date positions apart.")
else:
    print("\nDates not 5 trading-date positions apart:")
    print(gaps[gaps != 5])

Gap distribution between consecutive rebalance dates:
4.0      1
5.0    144
Name: count, dtype: int64

Dates not 5 trading-date positions apart:
end_date
2001-02-06    4.0
dtype: float64


However, based on the diagnostics we already ran, this particular check will report one gap = 4, for:

2001-01-30 -> 2001-02-06

even though we established that this period is valid. That's because all_dates contains only dates appearing somewhere as end_date, rather than a complete U.S. trading calendar.

So for your final notebook, I actually recommend a better check, which tests the thing you truly care about: whether the end price of one five-day period equals the starting price of the next period for stocks present in both periods.

Use this instead:

In [9]:
# Verify that consecutive retained formation dates represent
# consecutive non-overlapping 5-day holding periods

checks = []

for i in range(len(rebalance_dates) - 1):
    current_date = rebalance_dates[i]
    next_date = rebalance_dates[i + 1]

    current_period = (
        df[df["end_date"] == current_date]
        [["ticker", "close_future"]]
    )

    next_period = (
        df[df["end_date"] == next_date]
        [["ticker", "close_now"]]
    )

    comparison = current_period.merge(
        next_period,
        on="ticker",
        how="inner"
    )

    prices_match = np.isclose(
        comparison["close_future"],
        comparison["close_now"],
        rtol=1e-10,
        atol=1e-10
    )

    checks.append({
        "current_date": current_date,
        "next_date": next_date,
        "stocks_compared": len(comparison),
        "all_prices_match": prices_match.all()
    })

period_check = pd.DataFrame(checks)

print("Number of transitions checked:", len(period_check))
print("All periods consecutive:", period_check["all_prices_match"].all())

if not period_check["all_prices_match"].all():
    print("\nProblematic transitions:")
    print(period_check[~period_check["all_prices_match"]])

Number of transitions checked: 145
All periods consecutive: True


In [10]:
r = weekly_df["forward_return_5d"].dropna()

print(f"Fraction positive: {(r > 0).mean():.4%}")
print(f"Average positive return: {r[r > 0].mean():.4%}")
print(f"Average negative return: {r[r < 0].mean():.4%}")
print(f"Overall average return: {r.mean():.4%}")

Fraction positive: 50.7497%
Average positive return: 3.8324%
Average negative return: -4.1557%
Overall average return: -0.0864%


## 4. Assign stocks to 10 deciles on each rebalance date

On each `end_date`, I sort stocks by `pred_prob_up`:

- Decile 1 = lowest predicted probability of going up
- Decile 10 = highest predicted probability of going up

I use `pd.qcut()` to split the cross-section into 10 approximately equal-sized groups.

A small practical issue is that some stocks can have identical prediction values.  
To make `qcut()` stable, I first rank the predictions using `rank(method="first")`.

In [11]:
# Assign stocks to deciles separately on each formation date
weekly_df = weekly_df.copy()

weekly_df["decile"] = (
    weekly_df
    .groupby("end_date")["pred_prob_up"]
    .transform(
        lambda s: pd.qcut(
            s.rank(method="first"),
            q=10,
            labels=False
        ) + 1
    )
    .astype(int)
)

print(weekly_df.head(10))
print("\nNumber of rows:", len(weekly_df))

      ticker   end_date  close_now  close_future  pred_prob_up  \
0      10078 2001-01-30    32.0625       27.8125      0.513003   
146    10104 2001-01-30    30.3125       27.6250      0.510453   
292    10107 2001-01-30    63.3750       62.5625      0.440380   
437    10108 2001-01-30    49.9900       49.0600      0.490912   
756    10145 2001-01-30    47.3700       49.2500      0.520752   
902    10147 2001-01-30    78.7500       70.0000      0.468579   
1191   10299 2001-01-30    62.8125       57.6875      0.500677   
1337   10324 2001-01-30    87.7500       86.0625      0.510852   
1492   10401 2001-01-30    25.1000       23.8100      0.435851   
1638   10516 2001-01-30    14.9500       14.9800      0.508575   

      forward_return_5d  decile  
0             -0.132554       6  
146           -0.088660       5  
292           -0.012821       1  
437           -0.018604       3  
756            0.039688       7  
902           -0.111111       2  
1191          -0.081592       4  
1

In [12]:
print("\nColumns after assigning deciles:")
print(weekly_df.columns.tolist())


Columns after assigning deciles:
['ticker', 'end_date', 'close_now', 'close_future', 'pred_prob_up', 'forward_return_5d', 'decile']


## 5. Compute equal-weight decile returns on each rebalance date

For each formation date and decile, I take the simple average of the realized 5-day forward returns across all stocks in that decile.

This gives me one 5-day portfolio return for each decile on each rebalance date.

In [13]:
decile_returns_by_date = (
    weekly_df
    .groupby(["end_date", "decile"])["forward_return_5d"]
    .mean()
    .reset_index()
    .sort_values(["end_date", "decile"])
)

print(decile_returns_by_date.head(15))

     end_date  decile  forward_return_5d
0  2001-01-30       1          -0.044217
1  2001-01-30       2          -0.040760
2  2001-01-30       3          -0.023166
3  2001-01-30       4          -0.031394
4  2001-01-30       5          -0.016164
5  2001-01-30       6          -0.010452
6  2001-01-30       7          -0.011540
7  2001-01-30       8          -0.004308
8  2001-01-30       9          -0.022315
9  2001-01-30      10           0.013919
10 2001-02-06       1          -0.009277
11 2001-02-06       2          -0.004860
12 2001-02-06       3          -0.013028
13 2001-02-06       4           0.001886
14 2001-02-06       5          -0.017359


## 6. Put the decile returns into wide format and construct High-minus-Low

Now I reshape the data so that each column is one decile:

- column 1 = Low
- column 10 = High

Then I create the spread portfolio:

$$H-L = \text{Decile 10} - \text{Decile 1}$$

This is the same long-short spread that is reported in the papers.

In [14]:
decile_matrix = decile_returns_by_date.pivot(
    index="end_date",
    columns="decile",
    values="forward_return_5d"
).sort_index()

# Add High-minus-Low spread
decile_matrix["H-L"] = decile_matrix[10] - decile_matrix[1]

# Optional: rename columns for nicer display
decile_matrix = decile_matrix.rename(columns={1: "Low", 10: "High"})

print(decile_matrix.head())

decile           Low         2         3         4         5         6  \
end_date                                                                 
2001-01-30 -0.044217 -0.040760 -0.023166 -0.031394 -0.016164 -0.010452   
2001-02-06 -0.009277 -0.004860 -0.013028  0.001886 -0.017359 -0.021857   
2001-02-13 -0.020069 -0.020619 -0.021761 -0.031954 -0.041886 -0.041058   
2001-02-21 -0.007618  0.000866 -0.004936 -0.020523 -0.017615  0.003343   
2001-02-28 -0.007811  0.010933  0.023820  0.015927  0.013420  0.017898   

decile             7         8         9      High       H-L  
end_date                                                      
2001-01-30 -0.011540 -0.004308 -0.022315  0.013919  0.058136  
2001-02-06 -0.034499 -0.022411 -0.042647 -0.051665 -0.042389  
2001-02-13 -0.065539 -0.051960 -0.051398 -0.078588 -0.058519  
2001-02-21 -0.022735 -0.049395 -0.007080 -0.017302 -0.009684  
2001-02-28  0.025451  0.033592  0.052121  0.065874  0.073685  


## 7. Compute annualized return and annualized Sharpe ratio

Each row in `decile_matrix` is a **5-trading-day portfolio return**.

So I annualize using:

$$
\text{periods per year} = \frac{252}{5}
$$

Then:

$$
\text{Annualized Return} = \bar{r}_{5d} \times \frac{252}{5}
$$

$$
\text{Annualized Sharpe} = \frac{\bar{r}_{5d}}{\sigma_{5d}} \times \sqrt{\frac{252}{5}}
$$

This is the standard way to annualize fixed-horizon portfolio returns.

In [15]:
periods_per_year = 252 / 5  # 5-trading-day holding period

def annualized_stats(return_series):
    s = pd.Series(return_series).dropna()
    
    mean_5d = s.mean()
    std_5d = s.std(ddof=1)
    
    ann_return = mean_5d * periods_per_year
    
    if std_5d == 0 or np.isnan(std_5d):
        ann_sharpe = np.nan
    else:
        ann_sharpe = (mean_5d / std_5d) * np.sqrt(periods_per_year)
    
    return pd.Series({
        "Mean_5d_Return": mean_5d,
        "Std_5d_Return": std_5d,
        "Annualized_Return": ann_return,
        "Annualized_Sharpe": ann_sharpe,
        "N_periods": len(s)
    })

summary = decile_matrix.apply(annualized_stats, axis=0).T
summary

,Mean_5d_Return,Std_5d_Return,Annualized_Return,Annualized_Sharpe,N_periods
decile,,,,,
Low,-0.003838,0.024586,-0.193411,-1.108104,146.0
2,-0.003136,0.028030,-0.158034,-0.794158,146.0
3,-0.001772,0.027056,-0.089306,-0.464948,146.0
4,-0.001927,0.028469,-0.097137,-0.480620,146.0
5,-0.000695,0.028919,-0.035013,-0.170542,146.0
6,-0.000499,0.030976,-0.025133,-0.114289,146.0
7,-0.000635,0.028438,-0.031990,-0.158455,146.0
8,0.000370,0.031009,0.018639,0.084669,146.0
9,0.002870,0.035747,0.144636,0.569923,146.0


## 8. Make the final table look like the paper

To make the output easier to compare with the paper tables, I keep only the annualized return and annualized Sharpe ratio, and I convert the return to percent.

In [16]:
final_table = summary[["Annualized_Return", "Annualized_Sharpe", "N_periods"]].copy()
final_table["Annualized_Return_pct"] = final_table["Annualized_Return"] * 100

# Put columns in a nicer order
final_table = final_table[["Annualized_Return_pct", "Annualized_Sharpe", "N_periods"]]

# Optional: reorder rows
desired_order = ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"]
final_table = final_table.reindex([x for x in desired_order if x in final_table.index])

print(final_table.round(4))

        Annualized_Return_pct  Annualized_Sharpe  N_periods
decile                                                     
Low                  -19.3411            -1.1081      146.0
2                    -15.8034            -0.7942      146.0
3                     -8.9306            -0.4649      146.0
4                     -9.7137            -0.4806      146.0
5                     -3.5013            -0.1705      146.0
6                     -2.5133            -0.1143      146.0
7                     -3.1990            -0.1585      146.0
8                      1.8639             0.0847      146.0
9                     14.4636             0.5699      146.0
High                  23.6505             0.8721      146.0
H-L                   42.9915             1.9045      146.0


## 9. Interpretation of the output

The final table should be read as follows:

- `Low` is the decile with the lowest predicted probability of an up move.
- `High` is the decile with the highest predicted probability of an up move.
- `H-L` is a long-short strategy that buys the highest decile and shorts the lowest decile.
- `Annualized_Return_pct` is the annualized mean return in percent.
- `Annualized_Sharpe` is the annualized Sharpe ratio.

If the model is useful, I should generally see returns and Sharpe ratios improve as I move from `Low` to `High`.

In [17]:
# Average number of stocks in each decile
avg_names_per_decile = (
    weekly_df.groupby(["end_date", "decile"]).size()
    .groupby("decile")
    .mean()
)

print("Average number of stocks per decile:")
print(avg_names_per_decile.round(2))

# Check monotonicity visually
display_cols = [c for c in ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"] if c in decile_matrix.columns]
display(decile_matrix[display_cols].head())

Average number of stocks per decile:
decile
1     37.83
2     37.39
3     37.29
4     37.34
5     37.47
6     37.16
7     37.25
8     37.37
9     37.31
10    37.75
dtype: float64


decile,Low,2,3,4,5,6,7,8,9,High,H-L
end_date,,,,,,,,,,,
2001-01-30,-0.044217,-0.040760,-0.023166,-0.031394,-0.016164,-0.010452,-0.011540,-0.004308,-0.022315,0.013919,0.058136
2001-02-06,-0.009277,-0.004860,-0.013028,0.001886,-0.017359,-0.021857,-0.034499,-0.022411,-0.042647,-0.051665,-0.042389
2001-02-13,-0.020069,-0.020619,-0.021761,-0.031954,-0.041886,-0.041058,-0.065539,-0.051960,-0.051398,-0.078588,-0.058519
2001-02-21,-0.007618,0.000866,-0.004936,-0.020523,-0.017615,0.003343,-0.022735,-0.049395,-0.007080,-0.017302,-0.009684
2001-02-28,-0.007811,0.010933,0.023820,0.015927,0.013420,0.017898,0.025451,0.033592,0.052121,0.065874,0.073685


## Always-long equal-weight benchmark

As a reference benchmark, I also calculate the performance of an **always-long, equal-weight portfolio across all stocks**.

This benchmark does **not** sort stocks into deciles.  
Instead, on each portfolio formation date, it simply buys **all available stocks** and assigns each stock the same weight.

That is why this benchmark produces only **one portfolio return series**, and therefore only **one annualized return** and **one Sharpe ratio**.

### Step 1: Compute the 5-day forward return for each stock

For each stock $i$ on formation date $t$, the realized 5-day forward return is:

$$
r_{i,t} = \frac{\text{close\_future}_{i,t}}{\text{close\_now}_{i,t}} - 1
$$

where:

- $\text{close\_now}_{i,t}$ is the closing price at the portfolio formation date
- $\text{close\_future}_{i,t}$ is the closing price 5 trading days later

### Step 2: Compute the equal-weight benchmark return on each date

On each formation date $t$, the always-long benchmark return is the simple average of all stock returns on that date:

$$
r^{EW}_t = \frac{1}{N_t} \sum_{i=1}^{N_t} r_{i,t}
$$

where $N_t$ is the number of available stocks on date $t$.

So instead of creating 10 decile portfolios, I create only **one** portfolio each period:

- long all stocks
- equal weight each stock
- hold for 5 trading days

This gives a time series of benchmark returns:

$$
r^{EW}_{t_1}, r^{EW}_{t_2}, r^{EW}_{t_3}, \dots
$$

### Step 3: Annualize the mean return

Because each portfolio is held for 5 trading days, the number of holding periods per year is approximately:

$$
\frac{252}{5}
$$

where 252 is the standard number of trading days in a year.

The annualized return is therefore:

$$
\text{Annualized Return} = \bar{r}_{5d} \times \frac{252}{5}
$$

where $\bar{r}_{5d}$ is the average 5-day benchmark return across all periods.

### Step 4: Annualize the Sharpe ratio

The Sharpe ratio measures return relative to volatility.

Let $\sigma_{5d}$ denote the standard deviation of the 5-day benchmark returns.  
Then the annualized Sharpe ratio is:

$$
\text{Annualized Sharpe} = \frac{\bar{r}_{5d}}{\sigma_{5d}} \times \sqrt{\frac{252}{5}}
$$

### Why does this benchmark only give one return and one Sharpe ratio?

The decile analysis gives many values because stocks are split into many portfolios:

- Decile 1
- Decile 2
- ...
- Decile 10
- High-minus-Low

So each decile has its own return series and its own Sharpe ratio.

In contrast, the always-long benchmark does **not** split stocks into groups.  
It simply averages all stocks into one equal-weight portfolio on each date.

Therefore, it produces:

- one portfolio return series
- one annualized return
- one annualized Sharpe ratio

### Interpretation

This benchmark is useful because it shows how well a simple passive strategy performs without using the model.

If the model is useful in a long-only sense, then the **High decile** should ideally outperform this benchmark in terms of:

- annualized return
- Sharpe ratio

If the model is useful as a ranking model, then returns should generally improve from the **Low** decile to the **High** decile, and the **High-minus-Low** spread should be positive and economically meaningful.

In [18]:
# Always-long benchmark using the SAME sample as the decile portfolios

benchmark_df = weekly_df.copy()

benchmark_df["end_date"] = pd.to_datetime(benchmark_df["end_date"])
benchmark_df = benchmark_df.dropna(subset=["end_date", "forward_return_5d"])

ew_returns = (
    benchmark_df
    .groupby("end_date")["forward_return_5d"]
    .mean()
    .sort_index()
)

# Make sure it uses exactly the same dates as decile_matrix
ew_returns = ew_returns.reindex(decile_matrix.index).dropna()

periods_per_year = 252 / 5

annualized_return = ew_returns.mean() * periods_per_year
annualized_sharpe = (
    ew_returns.mean() / ew_returns.std(ddof=1)
) * np.sqrt(periods_per_year)

print("Always-long equal-weight benchmark, same sample as deciles")
print(f"Annualized return: {annualized_return:.4f}  ({annualized_return*100:.2f}%)")
print(f"Annualized Sharpe: {annualized_sharpe:.4f}")
print(f"Number of periods: {len(ew_returns)}")

Always-long equal-weight benchmark, same sample as deciles
Annualized return: -0.0231  (-2.31%)
Annualized Sharpe: -0.1180
Number of periods: 146


In [19]:
import pandas as pd
import numpy as np
from scipy import stats

# --------------------------------------------------
# t-test / p-value table for deciles
# Uses:
#   decile_matrix
# --------------------------------------------------

# Safe copy
test_df = decile_matrix.copy()

# Make sure dates are sorted
test_df.index = pd.to_datetime(test_df.index)
test_df = test_df.sort_index()

# Nice row order
row_order = [c for c in ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"] if c in test_df.columns]

rows = []
for col in row_order:
    s = pd.to_numeric(test_df[col], errors="coerce").dropna()

    # one-sample t-test: mean return = 0
    t_stat, p_val = stats.ttest_1samp(s, popmean=0.0, nan_policy="omit")

    rows.append({
        "Decile": str(col),
        "t": t_stat,
        "p": p_val
    })

t_table = pd.DataFrame(rows).set_index("Decile")
t_table = t_table.round(4)
t_table.columns = pd.MultiIndex.from_product([["All"], t_table.columns])

display(t_table)

All        
             t       p
Decile                
Low    -1.8860  0.0613
2      -1.3517  0.1786
3      -0.7913  0.4300
4      -0.8180  0.4147
5      -0.2903  0.7720
6      -0.1945  0.8460
7      -0.2697  0.7878
8       0.1441  0.8856
9       0.9700  0.3337
High    1.4843  0.1399
H-L     3.2415  0.0015

In [20]:
final_table = summary[["Annualized_Return", "Annualized_Sharpe", "N_periods"]].copy()
final_table["Annualized_Return_pct"] = final_table["Annualized_Return"] * 100

# Put columns in a nicer order
final_table = final_table[["Annualized_Return_pct", "Annualized_Sharpe", "N_periods"]]

# Optional: reorder rows
desired_order = ["Low", 2, 3, 4, 5, 6, 7, 8, 9, "High", "H-L"]
final_table = final_table.reindex([x for x in desired_order if x in final_table.index])

print(final_table.round(4))
#print(folders_old[folder_index])

        Annualized_Return_pct  Annualized_Sharpe  N_periods
decile                                                     
Low                  -19.3411            -1.1081      146.0
2                    -15.8034            -0.7942      146.0
3                     -8.9306            -0.4649      146.0
4                     -9.7137            -0.4806      146.0
5                     -3.5013            -0.1705      146.0
6                     -2.5133            -0.1143      146.0
7                     -3.1990            -0.1585      146.0
8                      1.8639             0.0847      146.0
9                     14.4636             0.5699      146.0
High                  23.6505             0.8721      146.0
H-L                   42.9915             1.9045      146.0
